# Knowledge Distillation - a big model teaches a small one

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/02_knowledge_distillation.ipynb)

**Goal.** Train a tiny student network and improve it with supervision from a much larger
teacher. The deployment architecture does not change: knowledge distillation (KD) aims to
increase student accuracy at the same parameter count, model size, and inference latency.

This notebook has two layers:

1. **Core lab:** teacher, no-KD student, and default KD student (`T=4`, `alpha=0.7`) are
   trained for the full budget.
2. **Compact experiment:** seven unique configurations vary one factor at a time. A short
   validation sweep chooses one configuration, which is then retrained with the full budget.

The experiment avoids a long Cartesian grid while still covering low, middle, and extreme
values of temperature and teacher-loss weight.

**Runtime:** set `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.

## 1. Setup

On Colab `torch`/`torchvision` are pre-installed, so nothing to `pip install`.

In [ ]:
import io, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T

GLOBAL_SEED = 0
STUDENT_SEED = 17
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- full-training budget ---
EPOCHS = 15
BATCH_SIZE = 128
LR = 0.05

# --- compact hyperparameter-search budget ---
SWEEP_EPOCHS = 3
SWEEP_SUBSET = 20_000

# Default configuration required by the core lab.
T_KD = 4.0
ALPHA_KD = 0.7

## 2. Data - train, validation, and test

MNIST contains 60,000 training images and 10,000 test images. We deterministically split
the original training set into 50,000 training and 10,000 validation examples.

The short sweep is selected by **validation accuracy**. The test set is reserved for final
reporting, which avoids choosing `T` and `alpha` using test performance.

In [ ]:
mean = (0.1307,)
std = (0.3081,)

train_tf = T.Compose([
    T.RandomCrop(28, padding=2),
    T.ToTensor(),
    T.Normalize(mean, std),
])
eval_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

# Two views of the same 60,000 training images: augmented for training, fixed for validation.
full_train_aug = torchvision.datasets.MNIST(
    "./data", train=True, download=True, transform=train_tf
)
full_train_eval = torchvision.datasets.MNIST(
    "./data", train=True, download=True, transform=eval_tf
)
test_set = torchvision.datasets.MNIST(
    "./data", train=False, download=True, transform=eval_tf
)

split_generator = torch.Generator().manual_seed(GLOBAL_SEED)
indices = torch.randperm(len(full_train_aug), generator=split_generator).tolist()
train_indices, val_indices = indices[:50_000], indices[50_000:]

train_set = Subset(full_train_aug, train_indices)
val_set = Subset(full_train_eval, val_indices)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)

# A fixed subset makes the seven exploratory runs inexpensive and directly comparable.
sweep_set = Subset(full_train_aug, train_indices[:SWEEP_SUBSET])

def make_train_loader(dataset, seed=GLOBAL_SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        generator=generator,
    )

train_loader = make_train_loader(train_set)
print(
    f"{len(train_set)} train / {len(val_set)} validation / {len(test_set)} test; "
    f"sweep subset = {len(sweep_set)}"
)

## 3. Teacher and student architectures

KD needs a clear capacity gap:

- **Teacher:** six convolution blocks in three stages with `64 -> 128 -> 256` channels.
- **Student:** three convolution blocks with `16 -> 32 -> 32` channels.

Only the student is deployed. The teacher is used during training and discarded afterward.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class Teacher(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        c1, c2, c3 = 64, 128, 256
        self.features = nn.Sequential(
            conv_block(1, c1), conv_block(c1, c1), nn.MaxPool2d(2),
            conv_block(c1, c2), conv_block(c2, c2), nn.MaxPool2d(2),
            conv_block(c2, c3), conv_block(c3, c3), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c3, num_classes)
        )

    def forward(self, x):
        return self.head(self.features(x))


class Student(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 16), nn.MaxPool2d(2),
            conv_block(16, 32), nn.MaxPool2d(2),
            conv_block(32, 32), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.head(self.features(x))

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


n_teacher = count_params(Teacher())
n_student = count_params(Student())
print(f"teacher params: {n_teacher:,}")
print(f"student params: {n_student:,}")
print(f"student parameter compression: {n_teacher / n_student:.1f}x")

## 4. Training, evaluation, and metric utilities

All students use the same optimizer, seed, initialization rule, and data split. Full runs
train on 50,000 examples; sweep runs train on the fixed 20,000-example subset.

Validation accuracy is printed after each epoch. Test accuracy is computed only after a
full model has been trained.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    was_training = model.training
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    if was_training:
        model.train()
    return 100.0 * correct / total


def train(model, loss_fn, epochs=EPOCHS, tag="", loader=None):
    """Train a model and return both the model and its validation-accuracy history."""
    loader = train_loader if loader is None else loader
    model.to(device).train()
    optimizer = torch.optim.SGD(
        model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = []

    for epoch in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), x, y)
            loss.backward()
            optimizer.step()
        scheduler.step()
        val_acc = evaluate(model, val_loader)
        history.append(val_acc)
        print(f"[{tag}] epoch {epoch + 1:02d}/{epochs} val_acc={val_acc:.2f}%")

    return model, history


@torch.no_grad()
def latency_ms(model, n=100):
    """Average batch-size-1 CPU inference latency in milliseconds."""
    model_cpu = copy.deepcopy(model).to("cpu").eval()
    x = torch.randn(1, 1, 28, 28)
    for _ in range(10):
        model_cpu(x)
    start = time.perf_counter()
    for _ in range(n):
        model_cpu(x)
    return (time.perf_counter() - start) / n * 1000.0


def model_size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6


def fresh_student():
    """Every experiment starts from the exact same student initialization."""
    torch.manual_seed(STUDENT_SEED)
    return Student()

## 5. Train the teacher

The teacher is trained normally with cross-entropy. It is then frozen and reused by every
KD experiment, so the seven sweep configurations see exactly the same teacher.

In [ ]:
torch.manual_seed(GLOBAL_SEED)
teacher = Teacher()
teacher, teacher_history = train(
    teacher,
    loss_fn=lambda out, x, y: F.cross_entropy(out, y),
    epochs=EPOCHS,
    tag="teacher",
    loader=make_train_loader(train_set, GLOBAL_SEED),
)
teacher_val_acc = evaluate(teacher, val_loader)
teacher_acc = evaluate(teacher, test_loader)
print(f"\nTeacher validation accuracy: {teacher_val_acc:.2f}%")
print(f"Teacher test accuracy:       {teacher_acc:.2f}%")

## 6. Baseline student - no distillation

The no-KD student uses only cross-entropy. It defines the accuracy that distillation must
beat without changing the deployed architecture.

In [ ]:
student_baseline = fresh_student()
student_baseline, baseline_history = train(
    student_baseline,
    loss_fn=lambda out, x, y: F.cross_entropy(out, y),
    epochs=EPOCHS,
    tag="student-no-kd",
    loader=make_train_loader(train_set, GLOBAL_SEED),
)
student_baseline_val_acc = evaluate(student_baseline, val_loader)
student_baseline_acc = evaluate(student_baseline, test_loader)
print(f"\nNo-KD student validation accuracy: {student_baseline_val_acc:.2f}%")
print(f"No-KD student test accuracy:       {student_baseline_acc:.2f}%")

## 7. Distillation loss

The student combines hard labels with the teacher's softened distribution:

$$
\mathcal{L} = \alpha T^2\,\mathrm{KL}
\left(\sigma(z_t/T)\,\|\,\sigma(z_s/T)\right)
+ (1-\alpha)\,\mathrm{CE}(z_s,y).
$$

- `T=1` keeps predictions sharp; larger `T` reveals more relative class information.
- Multiplication by `T**2` compensates for the gradient shrinkage caused by softening.
- `alpha=0` is label-only training; `alpha=1` is pure teacher supervision.

In [ ]:
def distillation_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=4.0,
    alpha=0.7,
):
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha must be in [0, 1]")

    ce_term = F.cross_entropy(student_logits, labels)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
    kd_term = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean",
    ) * (temperature ** 2)

    return alpha * kd_term + (1.0 - alpha) * ce_term

## 8. Core KD run - default configuration

The required KD student uses `T_KD=4.0` and `ALPHA_KD=0.7` for the full 15 epochs. The
teacher is frozen and evaluated under `torch.no_grad()`, so only the student is optimized.

In [ ]:
teacher.eval()
for parameter in teacher.parameters():
    parameter.requires_grad_(False)


def make_kd_loss_fn(temperature, alpha):
    def kd_loss_fn(student_out, x, y):
        # alpha=0 is the exact label-only control and can skip the teacher forward pass.
        if alpha == 0.0:
            return F.cross_entropy(student_out, y)
        with torch.no_grad():
            teacher_out = teacher(x)
        return distillation_loss(
            student_out,
            teacher_out,
            y,
            temperature=temperature,
            alpha=alpha,
        )
    return kd_loss_fn


def train_kd_student(temperature, alpha, epochs, dataset, tag):
    student = fresh_student()
    loader = make_train_loader(dataset, GLOBAL_SEED)
    return train(
        student,
        loss_fn=make_kd_loss_fn(temperature, alpha),
        epochs=epochs,
        tag=tag,
        loader=loader,
    )


student_kd, kd_default_history = train_kd_student(
    T_KD,
    ALPHA_KD,
    epochs=EPOCHS,
    dataset=train_set,
    tag=f"student-kd-T{T_KD:g}-a{ALPHA_KD:g}",
)
student_kd_val_acc = evaluate(student_kd, val_loader)
student_kd_acc = evaluate(student_kd, test_loader)
print(f"\nDefault KD validation accuracy: {student_kd_val_acc:.2f}%")
print(f"Default KD test accuracy:       {student_kd_acc:.2f}%")
print(f"KD test gain vs. no-KD:         {student_kd_acc - student_baseline_acc:+.2f}pp")

## 9. Compact sensitivity sweep

The sweep changes one factor at a time instead of evaluating every `T x alpha` pair:

- `T in [1, 2, 4, 8]` while `alpha=0.7`;
- `alpha in [0, 0.5, 0.7, 1]` while `T=4`.

The shared default pair is evaluated once, giving seven unique configurations. Each uses
the same initialization and a three-epoch, 20,000-example budget. `alpha=0` is the no-KD
control; the selected KD configuration must have `alpha>0`. These scores are for **ranking
configurations**, not for comparison with fully trained final models.

In [ ]:
T_VALUES = [1.0, 2.0, 4.0, 8.0]
ALPHA_VALUES = [0.0, 0.5, 0.7, 1.0]

sweep_configs = [
    {"temperature": temperature, "alpha": ALPHA_KD}
    for temperature in T_VALUES
]
sweep_configs += [
    {"temperature": T_KD, "alpha": alpha}
    for alpha in ALPHA_VALUES
    if alpha != ALPHA_KD
]

sweep_results = []
for config in sweep_configs:
    temperature = config["temperature"]
    alpha = config["alpha"]
    tag = f"sweep-T{temperature:g}-a{alpha:g}"
    sweep_student, sweep_history = train_kd_student(
        temperature,
        alpha,
        epochs=SWEEP_EPOCHS,
        dataset=sweep_set,
        tag=tag,
    )
    val_accuracy = evaluate(sweep_student, val_loader)
    sweep_results.append({
        "temperature": temperature,
        "alpha": alpha,
        "val_accuracy": val_accuracy,
    })
    del sweep_student

sweep_no_kd = next(
    row for row in sweep_results
    if row["temperature"] == T_KD and row["alpha"] == 0.0
)
for row in sweep_results:
    row["gain_vs_sweep_no_kd"] = row["val_accuracy"] - sweep_no_kd["val_accuracy"]

best_sweep = max(
    (row for row in sweep_results if row["alpha"] > 0.0),
    key=lambda row: row["val_accuracy"],
)

print(f"{'T':>6}{'alpha':>9}{'val accuracy':>16}{'vs sweep no-KD':>18}")
print("-" * 48)
for row in sweep_results:
    marker = " <- selected KD" if row is best_sweep else ""
    print(
        f'{row["temperature"]:>6.1f}{row["alpha"]:>9.1f}'
        f'{row["val_accuracy"]:>15.2f}%'
        f'{row["gain_vs_sweep_no_kd"]:>+17.2f}pp{marker}'
    )

print(
    f'\nSelected by the equal-budget sweep: T={best_sweep["temperature"]:g}, '
    f'alpha={best_sweep["alpha"]:g}'
)

## 10. Sweep plots

The two plots isolate the effect of each hyperparameter. The dashed line marks the
`alpha=0` control trained with the same subset, initialization, and three-epoch budget.
This keeps every comparison inside the sweep fair.

In [ ]:
import matplotlib.pyplot as plt

t_rows = [row for row in sweep_results if row["alpha"] == ALPHA_KD]
alpha_rows = [row for row in sweep_results if row["temperature"] == T_KD]
t_rows = sorted(t_rows, key=lambda row: row["temperature"])
alpha_rows = sorted(alpha_rows, key=lambda row: row["alpha"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(
    [row["temperature"] for row in t_rows],
    [row["val_accuracy"] for row in t_rows],
    marker="o",
    linewidth=2,
    color="#2a9d8f",
)
axes[0].set_xticks(T_VALUES)
axes[0].set_xlabel("temperature T (alpha=0.7)")
axes[0].set_ylabel("validation accuracy (%)")
axes[0].set_title("Temperature sensitivity")

axes[1].plot(
    [row["alpha"] for row in alpha_rows],
    [row["val_accuracy"] for row in alpha_rows],
    marker="o",
    linewidth=2,
    color="#457b9d",
)
axes[1].set_xticks(ALPHA_VALUES)
axes[1].set_xlabel("alpha (T=4)")
axes[1].set_ylabel("validation accuracy (%)")
axes[1].set_title("Teacher-weight sensitivity")

for axis in axes:
    axis.axhline(
        sweep_no_kd["val_accuracy"],
        color="#777777",
        linestyle="--",
        label="equal-budget no-KD control",
    )
    axis.grid(True, alpha=0.25)
    axis.legend()

plt.tight_layout()
plt.show()

## 11. Retrain the selected configuration

The selected pair is now trained from the same student initialization on all 50,000
training examples for 15 epochs. If the sweep selects the default pair, the already trained
default KD model is reused to avoid redundant computation.

In [ ]:
best_temperature = best_sweep["temperature"]
best_alpha = best_sweep["alpha"]
best_is_default = (
    best_temperature == T_KD and best_alpha == ALPHA_KD
)

if best_is_default:
    student_kd_best = student_kd
    kd_best_history = kd_default_history
    print("The sweep selected the default configuration; reusing its full-trained model.")
else:
    student_kd_best, kd_best_history = train_kd_student(
        best_temperature,
        best_alpha,
        epochs=EPOCHS,
        dataset=train_set,
        tag=f"student-best-T{best_temperature:g}-a{best_alpha:g}",
    )

student_kd_best_val_acc = evaluate(student_kd_best, val_loader)
student_kd_best_acc = evaluate(student_kd_best, test_loader)
print(f"Best KD validation accuracy: {student_kd_best_val_acc:.2f}%")
print(f"Best KD test accuracy:       {student_kd_best_acc:.2f}%")
print(f"Best KD gain vs. no-KD:      {student_kd_best_acc - student_baseline_acc:+.2f}pp")

## 12. Final comparison - deployment cost and accuracy

Teacher, no-KD student, default KD student, and selected KD student are compared using full
training. The teacher is a training-time tool. All student rows use exactly the same deployed
architecture, so KD can change accuracy but cannot change student size, parameter count, or
inference latency in a meaningful way.

In [ ]:
final_specs = [
    {
        "name": "Teacher",
        "model": teacher,
        "temperature": None,
        "alpha": None,
        "accuracy": teacher_acc,
    },
    {
        "name": "Student no-KD",
        "model": student_baseline,
        "temperature": None,
        "alpha": 0.0,
        "accuracy": student_baseline_acc,
    },
    {
        "name": "Student KD default",
        "model": student_kd,
        "temperature": T_KD,
        "alpha": ALPHA_KD,
        "accuracy": student_kd_acc,
    },
    {
        "name": "Student KD selected",
        "model": student_kd_best,
        "temperature": best_temperature,
        "alpha": best_alpha,
        "accuracy": student_kd_best_acc,
    },
]

final_results = []
for spec in final_specs:
    model = spec["model"]
    final_results.append({
        **spec,
        "params": count_params(model),
        "size_mb": model_size_mb(model),
        "latency_ms": latency_ms(model),
        "gain_pp": spec["accuracy"] - student_baseline_acc,
    })

print(
    f"{'model':<23}{'T':>6}{'alpha':>8}{'params':>12}{'size':>10}"
    f"{'test acc':>11}{'gain':>10}{'CPU latency':>14}"
)
print("-" * 94)
for row in final_results:
    t_text = "-" if row["temperature"] is None else f'{row["temperature"]:g}'
    a_text = "-" if row["alpha"] is None else f'{row["alpha"]:g}'
    print(
        f'{row["name"]:<23}{t_text:>6}{a_text:>8}'
        f'{row["params"]:>12,}{row["size_mb"]:>8.2f}MB'
        f'{row["accuracy"]:>10.2f}%{row["gain_pp"]:>+9.2f}pp'
        f'{row["latency_ms"]:>12.2f}ms'
    )

names = [row["name"] for row in final_results]
colors = ["#4c4c4c", "#d1495b", "#2a9d8f", "#457b9d"]
x = np.arange(len(final_results))

plot_metrics = [
    ("accuracy", "Test accuracy", "%"),
    ("gain_pp", "Accuracy gain vs. no-KD", "pp"),
    ("size_mb", "Model size", "MB"),
    ("latency_ms", "CPU latency / image", "ms"),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for axis, (key, title, unit) in zip(axes.flat, plot_metrics):
    values = [row[key] for row in final_results]
    bars = axis.bar(x, values, color=colors)
    axis.set_xticks(x, names, rotation=12)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.2)
    if key == "accuracy":
        axis.set_ylim(max(0, min(values) - 3), min(100, max(values) + 1))
    for bar, value in zip(bars, values):
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            value,
            f"{value:.2f}{unit}",
            ha="center",
            va="bottom" if value >= 0 else "top",
            fontsize=9,
        )

plt.suptitle("Knowledge distillation: quality gain at fixed student deployment cost")
plt.tight_layout()
plt.show()

teacher_row, baseline_row, default_row, selected_row = final_results
print("\nFINAL EVALUATION")
print("=" * 76)
print(
    f'The student is {teacher_row["params"] / baseline_row["params"]:.1f}x smaller '
    f'by parameters and {teacher_row["size_mb"] / baseline_row["size_mb"]:.1f}x smaller on disk.'
)
print(
    f'Default KD (T={T_KD:g}, alpha={ALPHA_KD:g}) changes test accuracy by '
    f'{default_row["gain_pp"]:+.2f}pp versus the no-KD student.'
)
print(
    f'The selected configuration (T={best_temperature:g}, alpha={best_alpha:g}) changes '
    f'test accuracy by {selected_row["gain_pp"]:+.2f}pp at the same student size and architecture.'
)
print(
    "Sweep scores rank hyperparameters under a short equal budget; the final full-training "
    "test result is the evidence used for the deployment comparison."
)

## 13. Takeaways

- KD transfers supervision, not architecture: every student has the same deployment cost.
- `T` controls how much class-similarity information is exposed. Very low values can be too
  sharp, while very high values can make the teacher signal too flat.
- `alpha` controls the source of supervision. The endpoints (`0` and `1`) are useful controls,
  while an intermediate value often balances labels and teacher knowledge.
- A short sweep is useful for ranking, but only the fully retrained selected model should be
  compared with the fully trained baselines.
- Hyperparameters are selected on validation data; test data is reserved for the final claim.

For edge deployment, distillation is commonly followed by quantization or pruning: first
improve the compact student's knowledge, then compress its representation or computation.